# 2.3 Random forest
In this exercise, you will implement your own version of a random forest, using the trees available from scikit-learn. You will then train the random forest using the MNIST dataset and assess its performance compared to decision trees.

# 1. Load the MNIST dataset into memory. Divide the 70,000 digits you have into a training set (60,000 digits) and a test set (10,000 digits).
- ≈85% training set

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

import kagglehub
import os

import pandas as pd

import numpy as np

In [ ]:
dataset = fetch_openml("mnist_784")
x = dataset ["data"]
y = dataset ["target"]

print(x)
print(y)

#### Not working, try this:

In [ ]:
dataset = fetch_openml("mnist_784", version=1, as_frame=False)
x = dataset["data"]
y = dataset["target"]

print(x.shape, y.shape)

#### Not working ... again
- try this:

In [ ]:
for attempt in range(5):
    try:
        print("Attempt:", attempt+1)
        dataset = fetch_openml("mnist_784", version=1, as_frame=False)
        print("Success!")
        break
    except Exception as e:
        print("Failed:", e)

# OK, OpenML servers are not collaborating --> just download the DS online and import it as a CSV

In [ ]:
# Download latest version
path = kagglehub.dataset_download("oddrationale/mnist-in-csv")

print("Path to dataset files:", path)
print(os.listdir(path))

In [ ]:
path_1 = os.path.join(path, "mnist_train.csv")
DF_train = pd.read_csv(path_1, sep=',')

path_2 = os.path.join(path, "mnist_test.csv")
DF_test = pd.read_csv(path_2, sep=',')

display(DF_train)
# display(DF_test)

# 1. Load the dataset from sklearn, as described in Subsec. 1.2.1. Then, based on your X and y, answer the following questions:
- How many records are available?
- Are there missing values?
- How many elements does each class contain?

In [ ]:
print(DF_train.shape)
print(DF_test.shape)

# How many records are available?
# 60k for train, 10k for test

print('----------------------')
# DF_train.info(show_counts = True)        # not working ??
print(DF_train.isna().sum())                # no missing values

print('----------------------')
# How many elements does each class contain?
print(DF_train['label'].value_counts())

In [ ]:
# from the train DF get x_train, y_train
# from the test DF get x_test, y_test
# BEFORE, since it's a DF we want a numpy array

# from DF to array:
version_1 = False
if version_1:
    x = DF_train.iloc[:, 1:]
    y = DF_train.iloc[:, 0]
    x = x.to_numpy
    y = y.to_numpy


x = DF_train.values[:, 1:]     # all rows, all columns except the first one
y = DF_train.values[:, 0]      # all rows, only the first column


# 2. Train a single decision tree (with the default parameters) on the training set, then compute its accuracy on the test set.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

clf = DecisionTreeClassifier()
clf.fit(x_train, y_train)
predictions = clf.predict(x_test)
training_predictions = clf.predict(x_train)
training_accuracy = accuracy_score(y_train, training_predictions)
accuracy = accuracy_score(predictions, y_test)

print(training_accuracy)
print(accuracy)

### 3. For this next exercise, you will implement your own version of a random forest. A random forest is an ensemble approach: it trains multiple trees on different portions of the dataset. This lowers the chance of overfitting on the dataset (the single tree might overfit its portion of data, but the overall “forest” will likely not).

#### - Each tree of the random forest is trained on N points extracted, *with replacement*, from the entire dataset --> estrai la pallina dall'urla **E LA RIMETTI DENTRO**

#### Bootstrap Sample
- When we say “bootstrap sample” in the context of Random Forests, it means that for each tree in the forest, you create a new dataset by randomly picking samples from the original dataset with replacement.

- With replacement means that after you pick a sample (a data point), you can pick it again (it’s like picking a card, putting it back in the deck, and picking again)

> #### - Note that, since the extraction of the points is done *with replacement*, selecting N points does not necessarily extract all points of the dataset. Indeed, only approximately 63.2% of all points will be extracted for each tree

## Important parameters:
#### - **max_features**
#### Each tree, additionally, bases each split decision using a subset of all features. The size of this subset, B, is often selected to be the square root of the total number of features available, but different random forest may adopt different values. This parameter can be defined for each decision tree through the **max_features** parameter.
- When building a tree, a random sample of max_features features will be extracted and used to select the split.

#### Another important parameter for random forests is the number of trees used. We will call this parameter n_estimators. During training, each of these trees (or estimators) is trained with its subset of data. During the prediction of a new list of points, each tree of the random forest will make its prediction. Then, through majority voting, the overall label assignment is made. Majority voting is just a fancy way of saying that the class selected by the highest number of trees is selected.
```python
class MyRandomForestClassifier():
    def __init__(self, n_estimators, max_features):
        pass

    # train the trees of this random forest using subsets of X (and y)
    def fit (self, X, y) :
        pass

    # predict the label for each point in X
    def predict (self, X):
        pass
```


In [ ]:
path_1 = os.path.join(path, "mnist_train.csv")
DF_train = pd.read_csv(path_1, sep=',')

path_2 = os.path.join(path, "mnist_test.csv")
DF_test = pd.read_csv(path_2, sep=',')
x = DF_train.values[:, 1:]     # all rows, all columns except the first one
y = DF_train.values[:, 0]      # all rows, only the first column

In [ ]:
class MyRandomForestClassifier():
    def __init__(self, n_estimators, max_features):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.trees = []
        pass

    # train the trees of this random forest using subsets of X (and y)
    def fit (self, x, y):
        '''
        - FOR EACH TREE, each one of them works on a subset of samples:
            - Select rows → bootstrap sample --> select all the rows, JUST WITH REPLACEMENT : put the card back in the deck
            - For those rows select also the corresponding class (same idx)
            
            Of these rows you
            
            - Select columns → feature subset for splits inside each tree --> select max_features columns for the train data --> NO REPLACEMENT
        '''
        
        self.trees = []
        # for each tree
        for tree in range(self.n_estimators):
            
            # get a bootstrap sample of x and y --> ROWS
            bootstrap_size = x.shape[0]
            bootstrap_idx = np.random.choice(x.shape[0], size=bootstrap_size, replace=True)

            feature_sample_idx = np.random.choice(x.shape[1], size=self.max_features, replace=False)

            # x_train_sample has the rows with index bootstrap_idx and ALL columns
            # from the bootstrapped matrix take all the rows but ONLY the columns with index = feature_sample_idx
            # x_train_sample = x[bootstrap_idx,feature_sample_idx] --> DOESN'T WORK WITH ARRAYS >:(
            x_train_sample = x[bootstrap_idx,:][:, feature_sample_idx]
            y_train_sample = y[bootstrap_idx]
            
            # create the classifier and train it
            clf = DecisionTreeClassifier()
            clf.fit(x_train_sample, y_train_sample)

            # STORE THE CLASSIFIER + column index that the classifier used for training
            self.trees.append((clf, feature_sample_idx))

        # trees = list of tuples (tree classifiers and their corresponding columns on which they trained on) --> each one of them trained on bootstrap_idx rows and feature_sample_idx columns. This subset is different for every tree, so each tree is unique.
        # why do I also pass the corresponding columns on which they trained on?
        # because later on during the predict when I'll feed the tree classifier x_test (which is 'complete') it has like 784 columns while
        # the tree classifier model expects only 50 (len(feature_sample_idx))
        # So I also pass feature_sample_idx so that I can slice the x_test later on to make it fit for the tree classifiers :)

        pass

    # predict the label for each point in X --> majority vote
    def predict (self, x):
        '''
        Majority vote among all tress:
            - for each tree in trees fix x_test so that it matches the tree classifier --> slice on feature_sample_idx
            - predict the chopped test_set
            - store the prediction of each tree
            - FOR EACH COLUMN (which is the first prediction), PICK THE MOST FREQUENT CLASS --> find the final list with all majority classes
        '''

        predictions = []
        for tree,feature_idx in self.trees:
            # reshape x_test according to feature_sample_idx of each tree
            x_test = x[:, feature_idx]

            # predict x_test for each tree and append the predictions
            prediction_per_tree = tree.predict(x_test)
            predictions.append(prediction_per_tree)
        
        predictions = np.array(predictions)     # shape: (n_trees, n_samples) 2D matrix

        # majority vote **PER COLUMN**
        majority_vote = []
        for idx_col in range(predictions.shape[1]):     # iterate on the columns
            col = predictions[:, idx_col]               # list of elements in the colum idx_col --> count each element and get the most frequent
        
            max_count_class = 0
            max_count = 0
            for el in np.unique(col):
                count = list(col).count(el)
                if count > max_count:
                    max_count = count
                    max_count_class = el
            majority_vote.append(max_count_class)
            majority_vote = [int(i) for i in majority_vote]
            
            # print(f"These are the column values: {col}")
            # print(f"This is the majority vote: {majority_vote}")

            # ignore
            cleaner_but_obscure_version = False
            if cleaner_but_obscure_version:
                from collections import Counter
                count = Counter(col)
                most_common_class, _ = count.most_common(1)[0]
        
        return majority_vote

if __name__ == '__main__':
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    number_of_trees = 10
    max_feature_per_tree = int((x_train.shape[1]) ** 0.5)
    random_classifier = MyRandomForestClassifier(number_of_trees, max_feature_per_tree)

    random_classifier.fit(x_train, y_train)
    majority_predictions = random_classifier.predict(x_test)
    
    # sanity check
    # print(majority_predictions)
    # print(len(majority_predictions))
    # print(x_test.shape)
    
    accuracy = accuracy_score(majority_predictions, y_test)
    print(f"The accuracy of the model with {number_of_trees} trees and {max_feature_per_tree} subset of features per tree is: {accuracy}")


#### cleaner version of the HOME-MADE RANDOM FOREST:

In [ ]:
class MyRandomForestClassifier():
    def __init__(self, n_estimators, max_features):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.trees = []
        pass

    # train the trees of this random forest using subsets of X (and y)
    def fit (self, x, y):
        '''
        - FOR EACH TREE, each one of them works on a subset of samples:
            - Select rows → bootstrap sample --> select all the rows, JUST WITH REPLACEMENT : put the card back in the deck
            - For those rows select also the corresponding class (same idx)
            
            Of these rows you
            
            - Select columns → feature subset for splits inside each tree --> select max_features columns for the train data --> NO REPLACEMENT
        '''
        
        self.trees = []
        # for each tree
        for tree in range(self.n_estimators):
            
            # get a bootstrap sample of x and y --> ROWS
            bootstrap_size = x.shape[0]
            bootstrap_idx = np.random.choice(x.shape[0], size=bootstrap_size, replace=True)

            feature_sample_idx = np.random.choice(x.shape[1], size=self.max_features, replace=False)

            x_train_sample = x[bootstrap_idx,:][:, feature_sample_idx]
            y_train_sample = y[bootstrap_idx]
            
            # create the classifier and train it
            clf = DecisionTreeClassifier()
            clf.fit(x_train_sample, y_train_sample)

            # STORE THE CLASSIFIER + column index that the classifier used for training
            self.trees.append((clf, feature_sample_idx))

        pass

    # predict the label for each point in X --> majority vote
    def predict (self, x):
        '''
        Majority vote among all tress:
            - for each tree in trees fix x_test so that it matches the tree classifier --> slice on feature_sample_idx
            - predict the chopped test_set
            - store the prediction of each tree
            - FOR EACH COLUMN (which is the first prediction), PICK THE MOST FREQUENT CLASS --> find the final list with all majority classes
        '''

        predictions = []
        for tree,feature_idx in self.trees:
            # reshape x_test according to feature_sample_idx of each tree
            x_test = x[:, feature_idx]

            # predict x_test for each tree and append the predictions
            prediction_per_tree = tree.predict(x_test)
            predictions.append(prediction_per_tree)
        
        predictions = np.array(predictions)     # shape: (n_trees, n_samples) 2D matrix

        # majority vote **PER COLUMN**
        majority_vote = []
        for idx_col in range(predictions.shape[1]):     # iterate on the columns
            col = predictions[:, idx_col]               # list of elements in the colum idx_col --> count each element and get the most frequent
        
            max_count_class = 0
            max_count = 0
            for el in np.unique(col):
                count = list(col).count(el)
                if count > max_count:
                    max_count = count
                    max_count_class = el
            majority_vote.append(max_count_class)
            majority_vote = [int(i) for i in majority_vote]
            
        return majority_vote

if __name__ == '__main__':
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    number_of_trees = 30
    max_feature_per_tree = int((x_train.shape[1]) ** 0.5)
    random_classifier = MyRandomForestClassifier(number_of_trees, max_feature_per_tree)

    random_classifier.fit(x_train, y_train)
    majority_predictions = random_classifier.predict(x_test)

    accuracy = accuracy_score(majority_predictions, y_test)
    print(f"The accuracy of the model with {number_of_trees} trees and {max_feature_per_tree} subset of features per tree is: {accuracy}")

# 4. Now train your random forest with the 60,000 points of the training set and compute its accuracy against the test set. How does the random forest behave? How does it compare to a decision tree? How does this performance vary as the number of estimators grow? Try values from 10 to 100 (with steps of 10) for n_estimators.

We partially already did this:
- train the model ✅
- compute test accuracy ✅

So the quest becomes:
> compare random forest test accuracy VS standard tree classifier test accuracy

> increase the number of trees (n_estimators) from 10 to 100 with steps of 10

In [ ]:
class MyRandomForestClassifier():
    def __init__(self, n_estimators, max_features):
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.trees = []
        pass

    # train the trees of this random forest using subsets of X (and y)
    def fit (self, x, y):
        '''
        - FOR EACH TREE, each one of them works on a subset of samples:
            - Select rows → bootstrap sample --> select all the rows, JUST WITH REPLACEMENT : put the card back in the deck
            - For those rows select also the corresponding class (same idx)
            
            Of these rows you
            
            - Select columns → feature subset for splits inside each tree --> select max_features columns for the train data --> NO REPLACEMENT
        '''
        
        self.trees = []
        # for each tree
        for tree in range(self.n_estimators):
            
            # get a bootstrap sample of x and y --> ROWS
            bootstrap_size = x.shape[0]
            bootstrap_idx = np.random.choice(x.shape[0], size=bootstrap_size, replace=True)

            feature_sample_idx = np.random.choice(x.shape[1], size=self.max_features, replace=False)

            # x_train_sample has the rows with index bootstrap_idx and ALL columns
            # from the bootstrapped matrix take all the rows but ONLY the columns with index = feature_sample_idx
            # x_train_sample = x[bootstrap_idx,feature_sample_idx] --> DOESN'T WORK WITH ARRAYS >:(
            x_train_sample = x[bootstrap_idx,:][:, feature_sample_idx]
            y_train_sample = y[bootstrap_idx]
            
            # create the classifier and train it
            clf = DecisionTreeClassifier()
            clf.fit(x_train_sample, y_train_sample)

            # STORE THE CLASSIFIER + column index that the classifier used for training
            self.trees.append((clf, feature_sample_idx))

        # trees = list of tuples (tree classifiers and their corresponding columns on which they trained on) --> each one of them trained on bootstrap_idx rows and feature_sample_idx columns. This subset is different for every tree, so each tree is unique.
        # why do I also pass the corresponding columns on which they trained on?
        # because later on during the predict when I'll feed the tree classifier x_test (which is 'complete') it has like 784 columns while
        # the tree classifier model expects only 50 (len(feature_sample_idx))
        # So I also pass feature_sample_idx so that I can slice the x_test later on to make it fit for the tree classifiers :)

        pass

    # predict the label for each point in X --> majority vote
    def predict (self, x):
        '''
        Majority vote among all tress:
            - for each tree in trees fix x_test so that it matches the tree classifier --> slice on feature_sample_idx
            - predict the chopped test_set
            - store the prediction of each tree
            - FOR EACH COLUMN (which is the first prediction), PICK THE MOST FREQUENT CLASS --> find the final list with all majority classes
        '''

        predictions = []
        for tree,feature_idx in self.trees:
            # reshape x_test according to feature_sample_idx of each tree
            x_test = x[:, feature_idx]

            # predict x_test for each tree and append the predictions
            prediction_per_tree = tree.predict(x_test)
            predictions.append(prediction_per_tree)
        
        predictions = np.array(predictions)     # shape: (n_trees, n_samples) 2D matrix

        # majority vote **PER COLUMN**
        majority_vote = []
        for idx_col in range(predictions.shape[1]):     # iterate on the columns
            col = predictions[:, idx_col]               # list of elements in the colum idx_col --> count each element and get the most frequent
        
            max_count_class = 0
            max_count = 0
            for el in np.unique(col):
                count = list(col).count(el)
                if count > max_count:
                    max_count = count
                    max_count_class = el
            majority_vote.append(max_count_class)
            
            # print(f"These are the column values: {col}")
            # print(f"This is the majority vote: {majority_vote}")

            # ignore
            cleaner_but_obscure_version = False
            if cleaner_but_obscure_version:
                from collections import Counter
                count = Counter(col)
                most_common_class, _ = count.most_common(1)[0]
        
        # just because numpy float are a pain in the ass
        majority_vote = [int(i) for i in majority_vote]
        return majority_vote

if __name__ == '__main__':
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

    number_of_trees = 10
    max_feature_per_tree = int((x_train.shape[1]) ** 0.5)
    random_classifier = MyRandomForestClassifier(number_of_trees, max_feature_per_tree)

    # random_classifier.fit(x_train, y_train)
    # majority_predictions = random_classifier.predict(x_test)
    
    # sanity check
    # print(majority_predictions)
    # print(len(majority_predictions))
    # print(x_test.shape)
    
    # accuracy = accuracy_score(majority_predictions, y_test)
    # print(f"The accuracy of the model with {number_of_trees} trees and {max_feature_per_tree} subset of features per tree is: {accuracy}")

    # compute accuracy of the standard tree classifier
    # clf = DecisionTreeClassifier()
    # clf.fit(x_train, y_train)
    # standard_tree_predictions = clf.predict(x_test)
    # print(f"The previous accuracy of the standard tree classifier is: {accuracy_score(standard_tree_predictions, y_test)}")

    
    number_of_trees = []
    for i in range(0, 102, 10):
        number_of_trees = i
        if number_of_trees == 0:
            number_of_trees = 1
        
        random_classifier = MyRandomForestClassifier(number_of_trees, max_feature_per_tree)
        random_classifier.fit(x_train, y_train)
        predictions = random_classifier.predict(x_test)
        accuracy = accuracy_score(predictions, y_test)
        print(f"Using {number_of_trees} trees, the accuracy of the model is: {accuracy}")


### Nice!
- As we would expect, the higher the number of trees, the higher the accuracy ... even though the increase in accuracy stays about the same after 30 trees 

# 5. Scikit-learn implements its own version of a random forest classifier, which is unsurprisingly called RandomForestClassifier (from sklearn.ensemble). Answer the same questions as the previous exercise. How does your implementation of the random forest compare to sklearn’s?

The questions to be answered:
- How does the random forest behave?
- How does it compare to a decision tree?
- How does this performance vary as the number of estimators grow?

> How does your implementation of the random forest compare to sklearn’s?

In [ ]:
path_1 = os.path.join(path, "mnist_train.csv")
DF_train = pd.read_csv(path_1, sep=',')

path_2 = os.path.join(path, "mnist_test.csv")
DF_test = pd.read_csv(path_2, sep=',')
x = DF_train.values[:, 1:]
y = DF_train.values[:, 0]

In [ ]:
x_test, x_train, y_test, y_train = train_test_split(x,y, test_size=0.2, random_state=42)

rnd_clf = RandomForestClassifier()
rnd_clf.fit(x_train, y_train)
predictions = rnd_clf.predict(x_test)
accuracy = accuracy_score(predictions, y_test)
print(f"The accuracy of the random forest classifier is: {accuracy}")

# How does your implementation of the random forest compare to sklearn’s?
Wild, not even with 100 of trees in our model we could achieve such high accuracy and by default the RandomForestClassifier uses ... 100 of them too! So actually we were quite close, apart from the fact that it would require our code like 40 seconds to run with 100 trees :)

# 6. Much like for decision trees, sklearn’s random forests can compute the importance of the features used.
# It does this by aggregating the feature importance of the trees into a single value.

- **Feature importance in decision trees**: In a decision tree, each feature’s importance can be computed based on how much it reduces impurity (like Gini impurity or entropy) during the splits. The feature that results in the greatest reduction of impurity is considered the most important. This means that, in a decision tree, the features that frequently split the data in the most impactful way are considered more important.

- **Random forests and feature importance**: Random forests consist of multiple decision trees. In the context of random forests, each tree is trained on a bootstrap sample (subset of data with replacement) and a random subset of features.

> **Example:**
Imagine you are classifying fruits, and you are deciding whether a fruit is an apple or a banana.  
- You first split the data based on “weight”.
- If you find that the split by weight (above or below a certain value) clearly separates apples from bananas, this is a strong split, and “weight” is considered an important feature.
- If “color” doesn’t help much in separating the apples from the bananas (e.g., you find that both apples and bananas are red, yellow, and green), “color” would be less important.

> In decision trees, the algorithm looks at every feature and calculates how much it reduces impurity when it is used to split the data. The more a feature reduces impurity, the more important it is.

#### What do we do with feature importance?
- In random forests, we have multiple decision trees. Each tree in the forest is trained with a bootstrap sample (randomly sampled data with replacement), and it uses a random subset of features (this is the max_features parameter).

- Each of these trees calculates feature importance the same way a decision tree does, by looking at how much each feature reduces impurity.

**So after training all the trees, we combine the feature importances from each tree to get the overall feature importance for each feature in the random forest**

$I_a = \frac{\sum_j I_{aj}}{\sum_i \sum_j I_{ij}}$

Where:  
- $I_aj$ = The feature importance of the a-th feature in the j-th tree.
- $I_a$ = The overall feature importance of the a-th feature in the random forest.
- $\sum_j I_{aj}$ = Sum of the importance of the a-th feature across all trees.
- $\sum_i \sum_j I_{ij}$ = The sum of feature importances for all features across all trees. This ensures that the total importance of all features in the forest adds up to 1.

> **Example:**
Imagine you have a forest with 3 trees, and 3 features: weight (feature 1), color (feature 2), and size (feature 3). Here’s how the feature importances might look:  

Tree 1:
- Weight: 0.6
- Color: 0.3
- Size: 0.1

Tree 2:
- Weight: 0.5
- Color: 0.4
- Size: 0.1

Tree 3:
- Weight: 0.4
- Color: 0.5
- Size: 0.1

> Now, we calculate the feature importance for each feature in the forest.
**For Weight (feature 1):**
- Sum of the feature importances across all trees = 0.6 + 0.5 + 0.4 = 1.5
- Sum of all feature importances (from all trees) = (0.6+0.3+0.1) + (0.5+0.4+0.1) + (0.4+0.5+0.1) = 4.0

> So, the overall importance of Weight is:

$I_{\text{Weight}} = \frac{1.5}{4.0} = 0.375$

#### To compute the feature importance of each tree, you can either use sklearn’s precomputed feature importance, **tree.feature_importances_**, or you can use your own implementation from Exercise 1 (aw hell nah).

In [ ]:
path = '../../Dataset/LAB4'
path_1 = os.path.join(path, "mnist_train.csv")
DF_train = pd.read_csv(path_1, sep=',')

path_2 = os.path.join(path, "mnist_test.csv")
DF_test = pd.read_csv(path_2, sep=',')
x = DF_train.values[:, 1:]
y = DF_train.values[:, 0]

In [ ]:
# split data into train and test
x_test, x_train, y_test, y_train = train_test_split(x,y, test_size=0.2, random_state=42)

# create a random forest
forest = RandomForestClassifier()

# train it and test it
forest.fit(x_train, y_train)
predictions = forest.predict(x_test)

# compute accuracy
accuracy = accuracy_score(predictions, y_test)
print(f"The accuracy of the random forest classifier is: {accuracy}")

# get feature importance
# list of NORMALIZED weights, one per feature. The weigth i-th is the importance of the i-th feature
# the weight is the ALREADY NORMALIZED importance of the feature across all trees --> sum the weigth for feature i-th among all trees / sum of the weigths of all features among all trees
# basiclaly this is already the result !!!
importances = forest.feature_importances_

verbose = False
for idx in range(x_train.shape[1]):
    current_feature = x_train[:, idx]
    current_weight = importances[idx]

    if verbose:
        if current_weight > 0:
            print(f'The feature number {idx}, which is: \n{current_feature}')
            print(f"Has weight {current_weight}")
            print()

# if you want the numerator and denominator separately you have to compute them by yourself!

all_weigths = []
for tree in forest.estimators_:
    # list of weights per feature OF ONE TREE
    weigthts_per_tree = tree.feature_importances_
    all_weigths.append(weigthts_per_tree)
# create a list with as many lists as there are trees and in each list there are as many weights as there are features

# get numerator and denominator
# denominator is always the same so compute it now
den = np.array(all_weigths)
den = den.sum()

# numerator
normalised_weight_per_feature = []
for idx in range(len(all_weigths[0])):
    num = []
    for w_tree in all_weigths:
        num.append(w_tree[idx])        
    
    num = np.array(num)
    num = num.sum()

    weight_per_feature = num / den
    normalised_weight_per_feature.append(weight_per_feature)

print(normalised_weight_per_feature)

# if you compare forest.feature_importances_ with normalised_weight_per_feature they should be identical


    




#### Easier way:

In [ ]:
# ok, DUH
importances = forest.feature_importances_

# this one is interesting:
all_weights = np.array([tree.feature_importances_ for tree in forest.estimators_])
numerator = all_weights.sum(axis=0)
denominator = all_weights.sum()
normalized = numerator / denominator